# Nemo Guardrails Testing

This notebook demonstrates how to test the GPT-20B model with and without Nemo Guardrails.

## Two Endpoints:
1. **Direct GPT Model** (via MaaS Gateway): Standard LLM inference without filtering
2. **Nemo Guardrails**: LLM with safety and policy guardrails enabled

## Prerequisites:
- GPT-20B model deployed in `models-as-a-service` namespace
- Nemo Guardrails deployed and configured
- Valid API token for authentication

In [ ]:
import requests
import json
import os
import subprocess
from urllib3.exceptions import InsecureRequestWarning
import warnings

# Suppress SSL warnings for demo
warnings.simplefilter('ignore', InsecureRequestWarning)

## Configuration

Set up endpoints and authentication

In [ ]:
# Auto-detect cluster domain
def get_cluster_domain():
    try:
        result = subprocess.run(
            ['oc', 'get', 'route', '-n', 'openshift-console', 'console', '-o', 'jsonpath={.spec.host}'],
            capture_output=True, text=True, check=True
        )
        console_host = result.stdout.strip()
        if 'apps.' in console_host:
            return console_host.split('apps.', 1)[1]
    except:
        pass
    return 'ocp.sandbox5548.opentlc.com'

CLUSTER_DOMAIN = get_cluster_domain()
print(f"✓ Cluster domain: {CLUSTER_DOMAIN}")

# Endpoints
NAMESPACE = "models-as-a-service"
MODEL_NAME = "gpt-oss-20b"

# Direct GPT endpoint (via MaaS Gateway)
GPT_DIRECT_URL = f"https://maas.apps.{CLUSTER_DOMAIN}/{NAMESPACE}/{MODEL_NAME}/v1/chat/completions"

# Nemo Guardrails endpoint
GUARDRAILS_URL = f"https://{MODEL_NAME}-nemo-guardrails-{NAMESPACE}.apps.{CLUSTER_DOMAIN}/v1/chat/completions"

print(f"\n📍 Endpoints:")
print(f"  Direct GPT:  {GPT_DIRECT_URL}")
print(f"  Guardrails:  {GUARDRAILS_URL}")

## Get API Token

Retrieve your authentication token

In [ ]:
# Option 1: Use token from environment variable
API_TOKEN = os.getenv('MODEL_API_TOKEN')

# Option 2: Get from Kubernetes secret
if not API_TOKEN:
    try:
        result = subprocess.run(
            ['oc', 'get', 'secret', f'{MODEL_NAME}-sa-token', '-n', NAMESPACE, '-o', 'jsonpath={.data.token}'],
            capture_output=True, text=True, check=True
        )
        import base64
        API_TOKEN = base64.b64decode(result.stdout.strip()).decode('utf-8')
        print("✓ Retrieved token from Kubernetes secret")
    except:
        print("⚠ Could not retrieve token automatically")
        API_TOKEN = input("Enter your API token: ").strip()

if API_TOKEN:
    print(f"✓ Token configured: {API_TOKEN[:20]}...{API_TOKEN[-10:]}")
else:
    print("❌ No token available - please set MODEL_API_TOKEN environment variable")

## Test Message

Simple question to test both endpoints

In [ ]:
# Test request payload (OpenAI-compatible format)
test_request = {
    "model": MODEL_NAME,  # Model name (not used by the endpoint but required by spec)
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of France? Answer in one sentence."
        }
    ],
    "max_tokens": 50,
    "temperature": 0.7
}

print("Test Question:")
print(f"  {test_request['messages'][0]['content']}")

## Test 1: Direct GPT Model (No Guardrails)

Call the model directly through the MaaS gateway

In [ ]:
headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

print("🚀 Calling Direct GPT Model...\n")

try:
    response = requests.post(
        GPT_DIRECT_URL,
        headers=headers,
        json=test_request,
        verify=False,
        timeout=30
    )
    
    print(f"Status Code: {response.status_code}")
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ Direct GPT Response:")
        print(f"  Model: {result.get('model', 'N/A')}")
        print(f"  Answer: {result['choices'][0]['message']['content']}")
        print(f"  Tokens: {result.get('usage', {}).get('total_tokens', 'N/A')}")
        print(f"  Finish Reason: {result['choices'][0].get('finish_reason', 'N/A')}")
    else:
        print(f"\n❌ Error:")
        print(response.text)
        
except Exception as e:
    print(f"\n❌ Exception: {e}")

## Test 2: GPT with Nemo Guardrails

Call the model through Nemo Guardrails for filtered/safe responses

In [ ]:
print("🛡️ Calling GPT through Nemo Guardrails...\n")

try:
    response = requests.post(
        GUARDRAILS_URL,
        headers=headers,
        json=test_request,
        verify=False,
        timeout=30
    )
    
    print(f"Status Code: {response.status_code}")
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ Guardrails Response:")
        print(f"  Model: {result.get('model', 'N/A')}")
        print(f"  Answer: {result['choices'][0]['message']['content']}")
        print(f"  Tokens: {result.get('usage', {}).get('total_tokens', 'N/A')}")
        print(f"  Finish Reason: {result['choices'][0].get('finish_reason', 'N/A')}")
        
        # Check if guardrails modified the response
        if 'guardrails' in result:
            print(f"\n  🛡️ Guardrails Info:")
            print(f"    {json.dumps(result['guardrails'], indent=4)}")
    else:
        print(f"\n❌ Error:")
        print(response.text)
        
except Exception as e:
    print(f"\n❌ Exception: {e}")

## Comparison

Compare both responses side-by-side

In [ ]:
print("\n" + "="*80)
print("COMPARISON: Direct GPT vs Nemo Guardrails")
print("="*80)
print("\nQuestion:")
print(f"  {test_request['messages'][0]['content']}")
print("\n" + "-"*80)
print("\n📍 DIRECT GPT (No filtering):")
print("  - Faster response")
print("  - No content filtering")
print("  - Direct access to model")
print("\n🛡️ NEMO GUARDRAILS (With safety filters):")
print("  - May add latency (guardrails processing)")
print("  - Content safety checks")
print("  - Policy enforcement")
print("  - Can block or modify unsafe content")
print("\n" + "="*80)

## Test with Potentially Unsafe Content

Test how guardrails handle edge cases

In [ ]:
# Example: Test with a more sensitive question
sensitive_request = {
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "user",
            "content": "Write a simple 'Hello World' program in Python."
        }
    ],
    "max_tokens": 100,
    "temperature": 0.7
}

print("Testing with code generation request...\n")
print(f"Question: {sensitive_request['messages'][0]['content']}\n")

# Test both endpoints
print("🛡️ Guardrails Response:")
try:
    response = requests.post(GUARDRAILS_URL, headers=headers, json=sensitive_request, verify=False, timeout=30)
    if response.status_code == 200:
        print(response.json()['choices'][0]['message']['content'])
    else:
        print(f"Error: {response.status_code} - {response.text}")
except Exception as e:
    print(f"Exception: {e}")

## Summary

### Endpoints:
- **Direct GPT**: `https://maas.apps.<cluster>/<namespace>/<model>/v1/chat/completions`
  - Fast, unfiltered responses
  - Standard OpenAI-compatible API
  
- **Nemo Guardrails**: `https://<model>-nemo-guardrails-<namespace>.apps.<cluster>/v1/chat/completions`
  - Safety and policy enforcement
  - May modify or block unsafe content
  - Adds processing latency

### Use Cases:
- **Use Direct GPT when**: You need fast responses and handle safety at application level
- **Use Guardrails when**: You need automated safety checks, policy enforcement, or content filtering

### Authentication:
Both endpoints require the same Bearer token authentication via MaaS.